# 🔍 Contagem de Pessoas RGBT com Par único de imagem

Este notebook foi criado para permitir a exploração direta, clara e passo a passo da contagem de pessoas com **um único par de imagens (Óptica RGB + Térmica)**.

---

## 🛠 Passo 1: Importar Bibliotecas Essenciais e Configurar o Caminho

In [ ]:
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

# Adiciona o diretório app/ ao path do Python para importar os módulos nativos
app_dir = Path('../').resolve()
if str(app_dir) not in sys.path:
    sys.path.insert(0, str(app_dir))

print(f'✅ Módulos importados com sucesso a partir de: {app_dir}')

## 📸 Passo 2: Carregar as Imagens Brutas de Entrada (Landing / Bronze)

Carregamos a imagem óptica Wide (`DJI_0789_W.JPG`) e a imagem térmica (`DJI_0790_T.JPG`).

In [ ]:
# Caminhos das mídias brutas da câmera DJI
rgb_path = app_dir / 'data/landing/DJI_0789_W.JPG'
thermal_path = app_dir / 'data/landing/DJI_0790_T.JPG'

# Carregar via OpenCV (formato BGR)
img_rgb_raw = cv2.imread(str(rgb_path))
img_th_raw = cv2.imread(str(thermal_path))

print(f'Resolução Nativa RGB (Wide 24mm): {img_rgb_raw.shape[1]}x{img_rgb_raw.shape[0]} px')
print(f'Resolução Nativa Térmica (40mm):  {img_th_raw.shape[1]}x{img_th_raw.shape[0]} px')

# Visualização lado a lado em RGB
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(img_rgb_raw, cv2.COLOR_BGR2RGB))
axes[0].set_title('1. Sensor Óptico RGB Bruto (Wide 24mm)')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_th_raw, cv2.COLOR_BGR2RGB))
axes[1].set_title('2. Sensor Térmico Bruto (40mm)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Pré-Processamento (Somente corte)

Nesta etapa transformamos somente a imagem RGB raw. O corte central mantém a proporção da imagem térmica; depois redimensionamos o recorte apenas para que as duas layers tenham exatamente o mesmo canvas e possam ser sobrepostas. A imagem térmica permanece inalterada.

In [ ]:
def center_crop_to_aspect(image, target_shape):
    """Corta o centro sem distorcer, usando a proporção do alvo."""
    target_h, target_w = target_shape[:2]
    height, width = image.shape[:2]
    target_aspect = target_w / target_h
    source_aspect = width / height

    if source_aspect > target_aspect:
        crop_h = height
        crop_w = int(round(height * target_aspect))
    else:
        crop_w = width
        crop_h = int(round(width / target_aspect))

    left = (width - crop_w) // 2
    top = (height - crop_h) // 2
    return image[top:top + crop_h, left:left + crop_w].copy()

# 1. Corte central RGB usando a proporção exata da imagem térmica.
img_rgb_cropped = center_crop_to_aspect(img_rgb_raw, img_th_raw.shape)

# 2. Canvas final exatamente igual ao da imagem térmica.
thermal_size = (img_th_raw.shape[1], img_th_raw.shape[0])
img_rgb_crop_exact = cv2.resize(img_rgb_cropped, thermal_size, interpolation=cv2.INTER_AREA)
img_th_same_size = img_th_raw.copy()

# 3. Sobreposição simples 50% RGB + 50% térmica.
blend_overlay = cv2.addWeighted(img_rgb_crop_exact, 0.5, img_th_same_size, 0.5, 0)

print(f'RGB após corte (proporção térmica): {img_rgb_cropped.shape[1]}x{img_rgb_cropped.shape[0]} px')
print(f'Canvas final RGB/Térmica:            {thermal_size[0]}x{thermal_size[1]} px')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for axis, image, title in zip(axes, [img_rgb_crop_exact, img_th_same_size, blend_overlay], ['RGB cortada', 'Térmica raw', 'Layers sobrepostas (50/50)']):
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()

## Salvar resultados para análise

Os arquivos são salvos dentro da pasta do notebook, sem alterar as imagens raw.

In [ ]:
notebook_dir = app_dir / 'notebooks'
output_dir = notebook_dir / 'outputs' / 'crop_only'
output_dir.mkdir(parents=True, exist_ok=True)

rgb_crop_output = output_dir / f'{rgb_path.stem}_rgb_crop_exact{rgb_path.suffix.lower()}'
thermal_output = output_dir / f'{thermal_path.stem}_thermal_raw{thermal_path.suffix.lower()}'
overlay_output = output_dir / f'{rgb_path.stem}_thermal_overlay{rgb_path.suffix.lower()}'

assert cv2.imwrite(str(rgb_crop_output), img_rgb_crop_exact)
assert cv2.imwrite(str(thermal_output), img_th_same_size)
assert cv2.imwrite(str(overlay_output), blend_overlay)

print(f'RGB cortada: {rgb_crop_output}')
print(f'Térmica raw:  {thermal_output}')
print(f'Sobreposição: {overlay_output}')

## Verificação final

Confirme visualmente o arquivo de sobreposição antes de usar o par como dado de referência.

In [ ]:
print('RGB e térmica têm o mesmo tamanho:', img_rgb_crop_exact.shape == img_th_same_size.shape)
print('Arquivos exportados dentro de:', output_dir)